# Building Data Science Agents with LangChain

**Title:** Building Data Science Agents with LangChain  
**Difficulty:** Expert  
**Notebook:** 11 of 11  

---

> *Combining LLMs, Tools, Structured Output, Data, RAG, and Agents into autonomous Data Science assistants.*

This notebook teaches you how to build agents that can autonomously analyze datasets, generate visualizations, run statistical tests, and explain their findings -- all guided by natural language instructions.

## Learning Objectives

After this notebook you will be able to:

1. **Understand** the difference between traditional Data Science workflows and agent-based workflows
2. **Create** safe, validated tools for data analysis tasks
3. **Build** a Data Science agent that selects appropriate tools autonomously
4. **Combine** RAG, tools, and structured output in a single system
5. **Apply** security best practices for tool-using agents
6. **Compare** agent vs pipeline approaches and know when to use each

## Prerequisites

| Concept | Source |
|---------|--------|
| Tools and agents | Notebook 06 |
| Structured output (Pydantic) | Notebook 02 |
| RAG pipeline | Notebook 05 |
| Pandas and NumPy basics | Background |

> This notebook combines concepts from across the entire series.

## Setup

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from io import StringIO
from dotenv import load_dotenv
from typing import Literal

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

load_dotenv()
print('All imports loaded!')

In [ ]:
api_key = os.getenv('OPENAI_API_KEY', '')
ollama_available = False
try:
    import requests
    r = requests.get('http://localhost:11434/api/tags', timeout=2)
    ollama_available = r.status_code == 200
except: pass

print(f'OpenAI: {"available" if api_key else "NOT SET"}')
print(f'Ollama: {"available" if ollama_available else "NOT RUNNING"}')

---

## 1. What is a Data Science Agent?

### Traditional Workflow

```mermaid
graph TD
    Q[Question] --> H[Human Analyst]
    H --> P[Python Code]
    P --> A[Analysis]
    A --> V[Visualization]
    V --> C[Conclusion]
```

### Agent-Based Workflow

```mermaid
graph TD
    Q[Question] --> AG[DS Agent]
    AG --> PD[Plan/Decide]
    PD --> T1[Pandas Tool]
    PD --> T2[Statistics Tool]
    PD --> T3[Visualization Tool]
    PD --> T4[ML Tool]
    PD --> T5[RAG Tool]
    T1 --> R[Result]
    T2 --> R
    T3 --> R
    T4 --> R
    T5 --> R
    R --> E[Explanation]
```

### Key Difference

| Traditional | Agent |
|-------------|-------|
| Human decides every step | Agent decides which tools to use |
| Fixed code path | Dynamic tool selection |
| One question at a time | Can decompose complex questions |
| Requires Python knowledge | Natural language interface |

---

## 2. Sample Dataset

We create a realistic Data Science dataset for our agent to analyze.

In [ ]:
# Create a realistic dataset
np.random.seed(42)
n = 200

data = pd.DataFrame({
    'age': np.random.normal(35, 10, n).astype(int).clip(18, 65),
    'income': np.random.lognormal(10.5, 0.5, n).astype(int),
    'experience_years': np.random.exponential(5, n).clip(0, 30).astype(int),
    'education': np.random.choice(['Bachelor', 'Master', 'PhD'], n, p=[0.5, 0.35, 0.15]),
    'department': np.random.choice(['Engineering', 'Marketing', 'Sales', 'Data Science'], n),
    'performance_score': np.random.normal(75, 15, n).clip(0, 100).round(1),
    'projects_completed': np.random.poisson(5, n),
})

# Add some missing values
data.loc[data.sample(frac=0.05).index, 'income'] = np.nan
data.loc[data.sample(frac=0.03).index, 'performance_score'] = np.nan

# Save for agent
csv_path = Path('../data/employee_data.csv')
csv_path.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(csv_path, index=False)

print(f'Dataset: {data.shape[0]} rows x {data.shape[1]} columns')
print(f'Missing values: {data.isnull().sum().sum()}')
print()
print(data.head())

---

## 3. Data Science Tools

We create validated, safe tools for data analysis. Each tool is a `@tool` function.

### Tool Design Principles

| Principle | Implementation |
|-----------|----------------|
| **Input validation** | Check types, ranges, existence |
| **No arbitrary code** | Each tool does one specific thing |
| **Error handling** | Return clear error messages |
| **Bounded output** | Limit output size |
| **No side effects** | Tools do not modify the original data |

In [ ]:
# Global dataset store
_current_dataset = None
_dataset_name = None

@tool
def load_dataset(csv_content: str, name: str = 'dataset') -> str:
    """Load a CSV dataset for analysis. Provide the CSV content as a string."""
    global _current_dataset, _dataset_name
    try:
        _current_dataset = pd.read_csv(StringIO(csv_content))
        _dataset_name = name
        shape = _current_dataset.shape
        missing = _current_dataset.isnull().sum().sum()
        return f'Loaded {name}: {shape[0]} rows, {shape[1]} columns. Missing values: {missing}'
    except Exception as e:
        return f'Error loading dataset: {e}'

@tool
def dataset_summary() -> str:
    """Get a summary of the current dataset including shape, types, and basic stats."""
    if _current_dataset is None:
        return 'No dataset loaded. Use load_dataset first.'
    df = _current_dataset
    lines = [f'Dataset: {_dataset_name}', f'Shape: {df.shape}', '', 'Data types:']
    for col, dtype in df.dtypes.items():
        lines.append(f'  {col}: {dtype}')
    lines.append('')
    lines.append('Basic statistics:')
    lines.append(df.describe().to_string())
    return '\n'.join(lines)

@tool
def missing_value_analysis() -> str:
    """Analyze missing values in the current dataset."""
    if _current_dataset is None:
        return 'No dataset loaded.'
    df = _current_dataset
    missing = df.isnull().sum()
    if missing.sum() == 0:
        return 'No missing values found.'
    lines = ['Missing values:', '']
    for col in df.columns:
        count = missing[col]
        if count > 0:
            pct = count / len(df) * 100
            lines.append(f'  {col}: {count} ({pct:.1f}%)')
    return '\n'.join(lines)

@tool
def correlation_analysis() -> str:
    """Compute correlation matrix for numeric columns."""
    if _current_dataset is None:
        return 'No dataset loaded.'
    df = _current_dataset.select_dtypes(include=[np.number])
    if df.shape[1] < 2:
        return 'Need at least 2 numeric columns for correlation.'
    corr = df.corr()
    # Find strongest correlations
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            pairs.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)

    lines = ['Strongest correlations:', '']
    for c1, c2, val in pairs[:5]:
        strength = 'strong' if abs(val) > 0.7 else 'moderate' if abs(val) > 0.4 else 'weak'
        lines.append(f'  {c1} <-> {c2}: {val:.3f} ({strength})')
    return '\n'.join(lines)

@tool
def column_analysis(column_name: str) -> str:
    """Analyze a specific column in detail."""
    if _current_dataset is None:
        return 'No dataset loaded.'
    if column_name not in _current_dataset.columns:
        return f'Column {column_name} not found. Available: {list(_current_dataset.columns)}'
    col = _current_dataset[column_name]
    lines = [f'Column: {column_name}', f'Type: {col.dtype}', f'Non-null: {col.count()}/{len(col)}']
    if col.dtype in ['int64', 'float64']:
        lines.extend(['', 'Statistics:', f'  Mean: {col.mean():.2f}', f'  Median: {col.median():.2f}', f'  Std: {col.std():.2f}', f'  Min: {col.min():.2f}', f'  Max: {col.max():.2f}'])
    else:
        lines.extend(['', 'Value counts:', ''])
        for val, count in col.value_counts().head(10).items():
            lines.append(f'  {val}: {count}')
    return '\n'.join(lines)

@tool
def group_analysis(group_by: str, agg_column: str = None) -> str:
    """Analyze data grouped by a categorical column."""
    if _current_dataset is None:
        return 'No dataset loaded.'
    if group_by not in _current_dataset.columns:
        return f'Column {group_by} not found.'
    df = _current_dataset
    grouped = df.groupby(group_by)

    if agg_column and agg_column in df.columns and df[agg_column].dtype in ['int64', 'float64']:
        stats = grouped[agg_column].agg(['count', 'mean', 'median', 'std'])
        return f'Group by {group_by}, aggregated on {agg_column}: {stats.to_string()}'
    counts = grouped.size().to_dict()
    return f'Group by {group_by}: {json.dumps(counts, indent=2)}'

@tool
def suggest_model(task_type: str) -> str:
    """Suggest appropriate ML models for a given task type."""
    suggestions = {
        'classification': 'Logistic Regression, Random Forest, XGBoost, SVM',
        'regression': 'Linear Regression, Ridge, Random Forest, Gradient Boosting',
        'clustering': 'K-Means, DBSCAN, Hierarchical Clustering',
        'anomaly detection': 'Isolation Forest, DBSCAN, Autoencoders',
    }
    task = task_type.lower()
    for key, models in suggestions.items():
        if key in task:
            return f'For {key}: {models}'
    return f'Suggestions for {task_type}: Try multiple models and compare.'

print('8 Data Science tools created.')
print('Tools: load_dataset, dataset_summary, missing_value_analysis,')
print('       correlation_analysis, column_analysis, group_analysis, suggest_model')

---

## 4. Building the Data Science Agent

The agent receives a natural language question and decides which tools to use.

```mermaid
graph TD
    Q[User Question] --> LLM[LLM Agent]
    LLM --> TD[Tool Decision]
    TD --> T1[Load Data]
    TD --> T2[Summary]
    TD --> T3[Correlation]
    TD --> T4[Group Analysis]
    TD --> T5[Missing Values]
    T1 --> R[Results]
    T2 --> R
    T3 --> R
    T4 --> R
    T5 --> R
    R --> LLM2[LLM Explanation]
    LLM2 --> A[Final Answer]
```

In [ ]:
# Agent with explicit tool-calling workflow
class DataScienceAgent:
    """Agent that analyzes datasets using safe tools."""

    def __init__(self, llm):
        self.llm = llm
        self.tools = [
            load_dataset, dataset_summary, missing_value_analysis,
            correlation_analysis, column_analysis, group_analysis, suggest_model
        ]
        self.tool_map = {t.name: t for t in self.tools}

    def _build_tool_description(self):
        return '\n'.join([
            f'- {t.name}: {t.description}' for t in self.tools])

    def analyze(self, question):
        tool_desc = self._build_tool_description()

        prompt = ChatPromptTemplate.from_messages([
            ('system', system_msg),
            ('human', '{question}')])

        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({'question': question})

    def interactive_analyze(self, question, steps=3):
        """Multi-step analysis: gather info, then explain."""
        tool_desc = self._build_tool_description()

        # Step 1: Decide which tools to use
        plan_prompt = ChatPromptTemplate.from_messages([
            ('system', f'You are a DS planner. Given a question and available tools, list the tools to use in order. Tools:\n{tool_desc}\n\nReturn ONLY tool names, one per line.'),
            ('human', '{question}')])
        plan = (plan_prompt | self.llm | StrOutputParser()).invoke({'question': question})

        # Step 2: Execute tools
        results = []
        for line in plan.strip().split(chr(10)):
            tool_name = line.strip().split('(')[0].split(':')[0].strip().lower()
            # Find matching tool
            for name, t in self.tool_map.items():
                if name.lower() in tool_name.lower() or tool_name.lower() in name.lower():
                    try:
                        result = t.invoke({})
                        results.append(f'{name}: {result}')
                    except Exception as e:
                        results.append(f'{name}: Error - {e}')
                    break

        # Step 3: Explain
        explain_prompt = ChatPromptTemplate.from_messages([
            ('system', 'You are a DS analyst. Explain the findings from the tool results clearly and concisely.'),
            ('human', 'Question: {question}\n\nTool Results:\n{results}\n\nExplanation:')])
        explanation = (explain_prompt | self.llm | StrOutputParser()).invoke({
            'question': question,
            'results': '\n\n'.join(results[:5])
        })
        return {'plan': plan, 'tool_results': results, 'explanation': explanation}

# Create agent
agent = DataScienceAgent(ChatOpenAI(model='gpt-4o-mini', temperature=0))
print('Data Science Agent ready!')

In [ ]:
# Load the dataset first
csv_content = data.to_csv(index=False)

# Load dataset
load_result = load_dataset.invoke({'csv_content': csv_content, 'name': 'employee_data'})
print(load_result)

In [ ]:
# Test the agent with different questions
questions = [
    'What are the key characteristics of this dataset?',
    'Is there a relationship between experience and income?',
    'How do different departments compare in performance?',
]

for q in questions:
    print(f'Q: {q}')
    result = agent.interactive_analyze(q)
    print(f'Plan: {result["plan"][:100]}...')
    print(f'Explanation: {result["explanation"][:200]}...')
    print()

---

## 5. Data Science Workflow with Agent

Let us walk through a complete Data Science analysis workflow.

```mermaid
graph TD
    L[Load Dataset] --> P[Profile]
    P --> MV[Missing Values]
    MV --> S[Statistics]
    S --> C[Correlations]
    C --> G[Group Analysis]
    G --> F[Findings]
```

In [ ]:
# Complete workflow
workflow = DataScienceAgent(ChatOpenAI(model='gpt-4o-mini', temperature=0))

# Load
print('Step 1: Loading dataset...')
load_dataset.invoke({'csv_content': csv_content, 'name': 'employees'})

# Profile
print('\nStep 2: Profiling...')
print(dataset_summary.invoke({})[:300])

# Missing values
print('\nStep 3: Missing values...')
print(missing_value_analysis.invoke({}))

# Statistics
print('\nStep 4: Key columns...')
print(column_analysis.invoke({'column_name': 'income'}))

# Correlations
print('\nStep 5: Correlations...')
print(correlation_analysis.invoke({}))

# Group analysis
print('\nStep 6: By department...')
print(group_analysis.invoke({'group_by': 'department', 'agg_column': 'performance_score'}))

---

## 6. API vs Ollama

| Aspect | OpenAI API | Ollama Local |
|--------|-----------|--------------|
| Tool selection | Excellent | Good (model-dependent) |
| Complex reasoning | Excellent | Varies |
| Cost | Per-token | Free |
| Privacy | Data sent to API | Local only |
| Speed | Network dependent | Hardware dependent |

In [ ]:
if ollama_available:
    local_agent = DataScienceAgent(ChatOllama(model='llama3.2', temperature=0))

    # Load dataset for local agent
    load_dataset.invoke({'csv_content': csv_content, 'name': 'employees'})

    result = local_agent.interactive_analyze('What is the average income?')
    print('Local agent result:')
    print(f'Explanation: {result["explanation"][:300]}')
else:
    print('Ollama not available. Run ollama serve first')

---

## 7. Agent vs Pipeline

### When to Use an Agent

| Scenario | Agent | Pipeline |
|----------|-------|----------|
| Exploratory analysis | Best | Not suitable |
| Unknown question type | Best | Not suitable |
| Multi-step reasoning | Good | Requires manual design |
| Repeatable reports | Risky (may vary) | Best (deterministic) |
| Production scoring | Not suitable | Best |
| Real-time dashboards | Not suitable | Best |

### Agent Limitations

- **Non-deterministic**: Same question may use different tools
- **Slower**: Multiple LLM calls for tool selection
- **Costly**: Each tool call may involve an LLM call
- **Less reliable**: May choose wrong tools or fail

> *Use agents for exploration and discovery. Use pipelines for production and repeatability.*

---

## 8. Security: Safe Data Science Agents

Data Science agents have unique security concerns because they work with data and code.

### Threats

| Threat | Description | Mitigation |
|--------|-------------|------------|
| **Arbitrary code** | Agent generates unrestricted Python | Use pre-defined tools only |
| **Data leakage** | Agent exposes sensitive data | Limit output size, mask PII |
| **Resource exhaustion** | Infinite loops or huge computations | Set iteration limits |
| **Privilege escalation** | Agent accesses beyond intended scope | Tool permissions, validation |
| **Model manipulation** | Adversarial data tricks the agent | Input validation, output filtering |

### Safe Architecture

```mermaid
graph TD
    U[User] --> Agent[DS Agent]
    Agent --> TD{Tool Decision}
    TD --> V[Validation]
    V --> T[Safe Tools]
    T --> R[Result]
    R --> F[Filter/Limit]
    F --> A[Answer]
```

### Security Checklist

1. **Pre-defined tools only** -- No `eval()`, `exec()`, or `subprocess`
2. **Input validation** -- Check column names, types, ranges
3. **Output limits** -- Cap result size and iteration count
4. **No raw file access** -- Tools cannot read arbitrary files
5. **Audit logging** -- Log all tool calls and results

In [ ]:
# Security best practices for DS agents
def create_safe_ds_agent(llm, max_iterations=5):
    """Create a DS agent with security constraints."""

    # Only allow safe tools
    safe_tools = [
        dataset_summary, missing_value_analysis, correlation_analysis,
        column_analysis, group_analysis, suggest_model
    ]
    tool_map = {t.name: t for t in safe_tools}

    class SafeDSAgent:
        def __init__(self):
            self.iterations = 0
            self.max_iter = max_iterations

        def analyze(self, question):
            self.iterations = 0
            tools_desc = '\n'.join(f'- {t.name}: {t.description}' for t in safe_tools)

            prompt = ChatPromptTemplate.from_messages([
                ('system', f'You are a DS analyst. Use ONLY these tools:\n{tools_desc}\nMax {max_iterations} steps. Be concise.'),
                ('human', '{question}')])
            return (prompt | llm | StrOutputParser()).invoke({'question': question})

    return SafeDSAgent()

# Create safe agent
safe_agent = create_safe_ds_agent(
    ChatOpenAI(model='gpt-4o-mini', temperature=0),
    max_iterations=3
)
print('Safe DS agent created with max 3 iterations.')

---

## 9. Exercises

### Exercise 1: Visualization Tool

Create a `create_visualization(column_name: str)` tool that generates a histogram or bar chart. Save it to `../data/viz.png` and return the file path. Hint: use `matplotlib`.

### Exercise 2: Outlier Detection Tool

Create an `outlier_detection(column_name: str)` tool that detects outliers using the IQR method. Return the number of outliers and their indices.

### Exercise 3: Multi-Dataset Comparison

Extend the agent to handle two datasets. Create a `compare_datasets()` tool that compares columns across two loaded datasets.

### Challenge 1: Streaming Analysis

Build an agent that streams its analysis process: first shows the plan, then executes each tool, and finally explains findings. Each step should be visible to the user.

### Challenge 2: Safe DS Agent with Logging

Build a complete safe Data Science Agent that logs every tool call, validates all inputs, limits output size, and provides an audit trail of its analysis.

---

## 10. Key Takeaways

| Concept | Summary |
|---------|---------|
| **DS Agent** | LLM that autonomously selects data analysis tools |
| **Safe tools** | Pre-defined, validated, no arbitrary code |
| **Multi-step** | Agent plans, executes, and explains |
| **Agent vs Pipeline** | Agent for exploration, pipeline for production |
| **Security** | Validate inputs, limit outputs, log actions |

### The Complete LangChain Data Science Stack

```
01. Introduction         -- What is LangChain?
02. Models & Prompts     -- Chat models, messages, templates
03. LCEL & Chains        -- Pipeline composition
04. Embeddings           -- Vector representations
05. Basic RAG            -- Retrieval-augmented generation
06. Tools & Agents       -- Dynamic tool calling
07. Capstone             -- Data Science AI Tutor
08. Advanced RAG         -- Chunking, reranking, evaluation
09. Document Processing  -- PDFs, CSVs, JSON, multimodal
10. SQL & Databases      -- Natural language database querying
11. DS Agents            -- Autonomous data analysis
```

> *You now have a complete toolkit for building AI-powered Data Science applications with LangChain.*